# Stage 3.B — distributed aero-first BO campaign (RE-RUN, coordination fixed)

**Why a re-run:** the first attempt fragmented — Google Drive made duplicate `blade_campaign` folders when the 3 sessions raced to create it, so they never shared one ledger. This version **fixes that**: cell 4 (PREP) creates ONE fresh folder (`campaign_run2`) and seeds it with ALL designs from the broken run, and cell 5 (GUARD) refuses to start unless exactly one shared folder exists. So we **continue** from the prior work toward 300 designs of *pooled* learning — nothing is wasted.

**How to launch (order matters):** run **Session 0 first** through the RUN cell (its PREP seeds the folder). Once it's going, run **Sessions 1 & 2** — their GUARD waits for Session 0's folder to sync, then they join. Set only `SESSION_INDEX` (0/1/2) per session.

**Async, verified:** a seconds-long pre-flight (cell 6) smoke-tests the async loop before the real run, and the RUN cell streams a LIVE per-completion async verdict. Resumable: re-running RUN after a Colab drop continues from the ledger.

## 1. Repo + deps + SU2 + Drive

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")   # 1 thread/worker -> N processes on N cores

IN_COLAB = importlib.util.find_spec("google.colab") is not None
BRANCH = "main"  # the Stage-3 campaign machinery is merged to main
REPO = Path("/content/fan-optimization") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not REPO.exists():
        subprocess.run(["git", "clone", "-b", BRANCH,
                        "https://github.com/clingergab/fan-optimization.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "origin", BRANCH], check=True)
    subprocess.run("apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
                   "libxft2 libxinerama1 unzip".split(), check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[bo]"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gmsh", "cadquery"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/fanopt")
else:
    DRIVE_ROOT = REPO / "data"
for p in (str(REPO), str(REPO / "src"), str(REPO / "scripts")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("repo:", REPO, "| drive:", DRIVE_ROOT)

In [ ]:
import urllib.request
from fanopt.cfd.phase3 import find_su2
SU2_BIN = find_su2()
if SU2_BIN is None and IN_COLAB:
    LOCAL = Path("/content/su2")
    if not any(LOCAL.rglob("SU2_CFD")):
        zc = DRIVE_ROOT / "su2" / "SU2-v8.0.1-linux64.zip"
        if not zc.exists():
            zc.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(
                "https://github.com/su2code/SU2/releases/download/v8.0.1/SU2-v8.0.1-linux64.zip", str(zc))
        LOCAL.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(zc), "-d", str(LOCAL)], check=True)
    hit = next(LOCAL.rglob("SU2_CFD"), None)
    if hit: subprocess.run(["chmod", "+x", str(hit)], check=False)
    SU2_BIN = str(hit) if hit else None
assert SU2_BIN, "SU2 not found"
print("SU2:", SU2_BIN)

## 2. Campaign config (edit `SESSION_INDEX` per session)

In [ ]:
# ---- EDIT PER SESSION -------------------------------------------------------------------
SESSION_INDEX = 0        # 0 in the FIRST session, 1 in the second, 2 in the third
N_SESSIONS    = 3        # how many Colab sessions you are running in total
SESSION_ID    = f"colab-{SESSION_INDEX}"
# ---- SHARED (identical in every session) ------------------------------------------------
BUDGET     = 300         # stop when the SHARED ledger reaches this many evaluations
N_INIT     = 24          # cold-start Sobol DoE (sliced round-robin across sessions)
N_WORKERS  = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
BATCH_SIZE = N_WORKERS   # match the core count so no worker sits idle (q per session; 3x = total in-flight)
# Fidelity — LOCKED coarse tier from the Stage-2 (2a) probe: coarse preserves the fine ranking
# (Kendall tau=1.0), so the campaign explores on coarse; fine (5,60) is reserved for the 3.C
# top-cluster confirmation. See ADR-0004.
N_CYCLES   = 3           # coarse campaign tier
INNER_ITER = 30
# FRESH folder for the fixed re-run — a unique name that does NOT collide with the corrupted
# blade_campaign* folders from the first attempt. Prep (cell 4) seeds it with all prior designs.
SHARED_DIR = DRIVE_ROOT / "campaign_run2"
print(f"session {SESSION_ID} of {N_SESSIONS} | {N_WORKERS} workers | budget {BUDGET} | shared {SHARED_DIR}")

## 3. PREP — seed the fresh shared folder from the first run (Session 0 only)

In [ ]:
# ============================================================================================
# PREP — creates the shared folder ONCE and seeds it with ALL designs from the first (broken) run,
# so we CONTINUE (not restart). Only Session 0 does this; Sessions 1 & 2 skip it and wait in cell 5.
# Safe to Run-All in every session (non-0 sessions no-op here). Idempotent (re-runs skip).
# ============================================================================================
import json
from fanopt.bo.campaign_analysis import find_shards, load_rows
if SESSION_INDEX == 0:
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    seed = SHARED_DIR / "evaluations_seed.jsonl"
    if seed.exists():
        print("seed already present — skipping consolidation")
    else:
        old = [s for shs in find_shards(DRIVE_ROOT, "blade_campaign*").values() for s in shs]
        rows = load_rows(old)
        uniq = {}
        for r in rows:
            uniq.setdefault(r.get("design_hash"), r)
        # strip the first run's per-eval TELEMETRY (dispatch/finish times use that run's clock) so
        # the new run's live async check isn't poisoned; keep design vector + objectives for the GP.
        drop = ("dispatch_s", "finish_s", "inflight_at_dispatch")
        clean = [{k: v for k, v in r.items() if k not in drop} for r in uniq.values()]
        seed.write_text("\n".join(json.dumps(r) for r in clean) + "\n")
        print(f"seeded {len(clean)} unique designs from {len(old)} old shards -> {seed}")
else:
    print(f"Session {SESSION_INDEX}: PREP is Session-0 only — skipping (cell 5 will wait for the folder).")

## 4. Safety guard — one shared folder, no duplicates (every session)

In [ ]:
# ============================================================================================
# SAFETY GUARD (run in EVERY session) — waits for the single shared folder to exist, and REFUSES to
# start if Drive made duplicates (the bug that fragmented the first run). This is the fix.
# ============================================================================================
import time
for _ in range(60):  # wait up to 10 min for Session 0's folder+seed to sync to this VM
    dups = sorted(p for p in DRIVE_ROOT.glob(SHARED_DIR.name + "*") if p.is_dir())
    if (SHARED_DIR / "evaluations_seed.jsonl").exists():
        break
    print("waiting for the shared folder + seed from Session 0 ...", flush=True); time.sleep(10)
dups = sorted(p for p in DRIVE_ROOT.glob(SHARED_DIR.name + "*") if p.is_dir())
assert (SHARED_DIR / "evaluations_seed.jsonl").exists(), "no shared seed — run PREP in Session 0 first."
assert len(dups) == 1, f"DUPLICATE shared folders exist: {dups}\nDelete the extras in Drive before launching."
print(f"shared folder OK (exactly one): {SHARED_DIR}")

## 5. Async pre-flight — seconds, no CFD; if it fails, do not launch

In [ ]:
from fanopt.bo.distributed_campaign import preflight_async_check
# Smoke test with a FAST dummy objective (NO CFD) — confirms the async MACHINERY works in THIS Colab
# runtime BEFORE committing to the multi-hour campaign: the pool fills to N_WORKERS and REFILLS on
# completion (reaches 2*N_WORKERS unique, no duplicates). Takes ~1 min at 12 workers.
pf = preflight_async_check(n_workers=N_WORKERS)
ps = pf["per_session"]["preflight"]
print(f"async pre-flight (no CFD): peak_concurrency={ps['peak_concurrency']}/{N_WORKERS}  "
      f"refilled={pf['reached_budget']}  no_duplicates={pf['no_duplicates']}  passed={pf['passed']}")
print(f"  (smoke utilization={ps['utilization']:.0%} — LOW is EXPECTED here: the fast dummy objective "
      f"makes the serial GP proposal the bottleneck. Real async utilization is measured LIVE in the "
      f"run cell, where 2.8h evals dwarf the proposal.)")
assert pf["passed"], "ASYNC PRE-FLIGHT FAILED — pool didn't fill/refill on completion; do NOT launch."
print("OK — async dispatch-on-completion works in this runtime. Launch below; watch the run cell's "
      "live utilization verdict on the real objective.")

## 6. Run the session — streams a LIVE async verdict per completion (resumable)

In [ ]:
import time
import run_blade_campaign_distributed as campaign
import fanopt.geometry.blade_cad as blade_cad
from fanopt.bo.distributed_campaign import read_ledger
blade_cad.N_RADIAL_SECTIONS = 40  # the objective's geometry resolution (ADR-0004)

CLAIM_TTL = 6 * 3600  # must exceed the per-eval wall time — measured ~3.6-3.85h, so 6h with margin
argv = ["--shared-dir", str(SHARED_DIR), "--session-id", SESSION_ID,
        "--session-index", str(SESSION_INDEX), "--n-sessions", str(N_SESSIONS),
        "--budget", str(BUDGET), "--n-init", str(N_INIT), "--batch-size", str(BATCH_SIZE),
        "--n-workers", str(N_WORKERS), "--su2-bin", SU2_BIN, "--poll-seconds", "10",
        "--claim-ttl", str(CLAIM_TTL)]
if N_CYCLES is not None:   argv += ["--n-cycles", str(N_CYCLES)]
if INNER_ITER is not None: argv += ["--inner-iter", str(INNER_ITER)]
# Long-running + resumable: every eval is appended to the shared Drive ledger, so re-running
# this cell after a Colab drop resumes from the ledger (no work lost).
_n0 = len(read_ledger(SHARED_DIR)[0]); _t0 = time.time()
campaign.main(argv)
_dt_h = (time.time() - _t0) / 3600; _dn = len(read_ledger(SHARED_DIR)[0]) - _n0
print(f"\nthis session: {_dn} evals in {_dt_h:.2f} h"
      + (f"  ->  ~{_dt_h / max(_dn, 1) * N_WORKERS:.2f} h/eval wall" if _dn else ""))

## 7. Campaign summary — designs, averages, progression, best (safe anytime)

In [ ]:
from fanopt.bo.campaign_analysis import campaign_report
# Full campaign summary (safe anytime — during or after the run). Reads this ONE shared folder.
shards = [str(p) for p in SHARED_DIR.glob("evaluations_*.jsonl")]
r = campaign_report(shards)
print(f"unique designs: {r['unique_designs']}  (finite {r['finite']}, failed/NaN {r['failed_nan']})  sources={r['sources']}")
if r.get("j_fan"):
    print(f"J_fan  min/mean/max : {r['j_fan']['min']:+.2e} / {r['j_fan']['mean']:+.2e} / {r['j_fan']['max']:+.2e}")
    print(f"mass g min/mean/max : {r['mass_g']['min']:.0f} / {r['mass_g']['mean']:.0f} / {r['mass_g']['max']:.0f}")
    beat = "BEAT" if r["bo_best"] and r["sobol_best"] and r["bo_best"] > r["sobol_best"] else "did NOT beat"
    print(f"Sobol(DoE) best {r['sobol_best']:+.2e} | BO best {r['bo_best']:+.2e}  (BO {beat} the DoE)")
    print(f"new running-bests {r['progression']['n_new_bests']} over {r['finite']} evals | Pareto {r['pareto_count']}")
    print("top designs by J_fan:")
    for d in r["top_by_j_fan"]:
        print(f"  J_fan={d['j_fan']:+.2e}  mass={d['mass_g']:.0f}g  {d['source']:5}  {d['design_hash']}")

## 8. Progression plot

In [ ]:
import matplotlib.pyplot as plt
rb = r["progression"]["running_best"]
plt.figure(figsize=(8, 4)); plt.plot(range(1, len(rb) + 1), rb, marker=".")
plt.xlabel("evaluation # (time order)"); plt.ylabel("running-best J_fan")
plt.title("Optimization progression — rising = learning, flat = random")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()